# Crop Baseline — High-Quality Pill Crops for Classification

- 목적: detection bbox를 classification/OCR 친화적인 crop으로 변환
- 데이터: AI Hub single 경구약제 (`detection_single.csv`)
- 핵심 원칙: 원본 이미지 crop, 정사각형 margin crop, reflect padding
- bbox 소스: GT bbox 또는 YOLO `best.pt` 예측 bbox 선택 가능
- 기본 권장: 실제 파이프라인과 맞추기 위해 YOLO prediction bbox 사용


In [ ]:
!pip install ultralytics opencv-python-headless -q

from google.colab import auth
auth.authenticate_user()

import math
import os
import shutil
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from ultralytics import YOLO
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)


## 1. Manifest 로드 및 스키마 확인

- baseline과 동일하게 detection manifest를 기준으로 사용
- 이후 classification label은 `item_seq`를 사용


In [ ]:
MANIFEST_PATH = "gs://pilliot-raw-data-2026/04_manifests/detection_single.csv"
df = pd.read_csv(MANIFEST_PATH)

print(f"rows: {len(df):,}")
print("columns:", df.columns.tolist())
display(df.head(2))

print("\n--- zip별 annotation 수 ---")
display(df.groupby(['split_type', 'label_zip_name']).size().reset_index(name='count'))


## 2. Subset 선택

- 우선 YOLO baseline과 같은 subset으로 맞춤
- crop 품질 검증 후 zip 범위를 넓혀도 됨


In [ ]:
TRAIN_LABEL_ZIPS = ['TL_38_단일.zip']
VAL_LABEL_ZIPS = ['VL_1_단일.zip']

def label_to_image_zip(label_zip):
    return label_zip.replace('TL_', 'TS_').replace('VL_', 'VS_')

train_df = df[df['label_zip_name'].isin(TRAIN_LABEL_ZIPS)].copy()
val_df = df[df['label_zip_name'].isin(VAL_LABEL_ZIPS)].copy()

train_df['split'] = 'train'
val_df['split'] = 'validation'

print(f"train rows: {len(train_df):,}")
print(f"val rows:   {len(val_df):,}")
display(train_df[['split', 'label_zip_name', 'image_file', 'item_seq', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h']].head(3))


## 3. 원본 이미지 다운로드

- 반드시 원본 이미지에서 crop
- YOLO 입력용 640 letterbox 이미지에서 crop하지 않음


In [ ]:
!df -h /content

shutil.rmtree('/content/raw_images', ignore_errors=True)
shutil.rmtree('/content/crops', ignore_errors=True)


In [ ]:
GCS_BASE = "gs://pilliot-raw-data-2026/00_original_zip/aihub_576_oral_pill/single"

def download_and_extract(label_zips, split, subset_df):
    dst = Path(f"/content/raw_images/{split}")
    dst.mkdir(parents=True, exist_ok=True)

    for label_zip in label_zips:
        image_zip = label_to_image_zip(label_zip)
        gcs_src = f"{GCS_BASE}/{split}/images/{image_zip}"
        selected = set(subset_df.loc[subset_df['label_zip_name'] == label_zip, 'image_file'].tolist())

        if not selected:
            print(f"skip {label_zip}: no selected images")
            continue

        print(f"Downloading {image_zip} ...")
        ret = os.system(f'gcloud storage cp "{gcs_src}" /content/tmp_images.zip')
        if ret != 0 or not Path('/content/tmp_images.zip').exists():
            print(f"  ERROR: download failed for {image_zip}")
            continue

        extracted = 0
        with zipfile.ZipFile('/content/tmp_images.zip') as zf:
            for member in zf.namelist():
                fname = Path(member).name
                if fname in selected:
                    (dst / fname).write_bytes(zf.read(member))
                    extracted += 1

        os.remove('/content/tmp_images.zip')
        used_gb = shutil.disk_usage('/content').used / 1e9
        print(f"  extracted: {extracted:,} images  disk used: {used_gb:.1f} GB")

download_and_extract(TRAIN_LABEL_ZIPS, 'train', train_df)
download_and_extract(VAL_LABEL_ZIPS, 'validation', val_df)


## 4. BBox 소스 선택

- 학습은 다시 하지 않고, 이미 만든 `best.pt`로 raw image에 추론만 수행
- single dataset 기준으로 이미지당 최고 confidence bbox 1개 사용
- GT bbox crop과 YOLO prediction bbox crop을 모두 비교 가능
- 기본값은 실제 파이프라인에 맞춘 `BBOX_SOURCE='yolo'`


In [ ]:
BBOX_SOURCE = 'yolo'  # 'ground_truth' or 'yolo'
YOLO_WEIGHTS_PATH = '/content/drive/MyDrive/Conference/Pillot-personal/runs/yolo11n_single_baseline/weights/best.pt'
YOLO_IMGSZ = 640
YOLO_CONF = 0.25
YOLO_IOU = 0.70

def build_gt_crop_df(subset_df):
    gt_df = subset_df.reset_index(drop=True).copy()
    gt_df['source_bbox'] = 'ground_truth'
    gt_df['pred_conf'] = np.nan
    return gt_df

def build_yolo_crop_df(subset_df, split, weights_path=YOLO_WEIGHTS_PATH, imgsz=YOLO_IMGSZ, conf=YOLO_CONF, iou=YOLO_IOU):
    src_dir = Path(f'/content/raw_images/{split}')
    base_df = subset_df[['image_file', 'item_seq', 'label_zip_name']].drop_duplicates('image_file').reset_index(drop=True)

    if not Path(weights_path).exists():
        raise FileNotFoundError(f'YOLO weights not found: {weights_path}')

    image_paths = []
    image_lookup = {}
    missing = 0
    for _, row in base_df.iterrows():
        image_path = src_dir / row['image_file']
        if image_path.exists():
            image_paths.append(str(image_path))
            image_lookup[row['image_file']] = row
        else:
            missing += 1

    print(f'[{split}] source images for YOLO crop: {len(image_paths):,}  missing: {missing:,}')

    model = YOLO(weights_path)
    results = model.predict(image_paths, imgsz=imgsz, conf=conf, iou=iou, verbose=False)

    records = []
    no_det = 0
    for image_path, result in zip(image_paths, results):
        image_file = Path(image_path).name
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            no_det += 1
            continue

        confs = boxes.conf.detach().cpu().numpy()
        best_idx = int(np.argmax(confs))
        x1, y1, x2, y2 = boxes.xyxy[best_idx].detach().cpu().numpy().tolist()
        base_row = image_lookup[image_file]

        records.append({
            'split': split,
            'label_zip_name': base_row['label_zip_name'],
            'image_file': image_file,
            'item_seq': base_row['item_seq'],
            'bbox_x': float(x1),
            'bbox_y': float(y1),
            'bbox_w': float(max(0.0, x2 - x1)),
            'bbox_h': float(max(0.0, y2 - y1)),
            'source_bbox': 'yolo_pred',
            'pred_conf': float(confs[best_idx]),
        })

    pred_df = pd.DataFrame(records)
    print(f'[{split}] YOLO detections kept: {len(pred_df):,}  no detection: {no_det:,}')
    if len(pred_df):
        display(pred_df.head(3))
    return pred_df

def resolve_crop_source_df(subset_df, split):
    if BBOX_SOURCE == 'ground_truth':
        return build_gt_crop_df(subset_df)
    if BBOX_SOURCE == 'yolo':
        return build_yolo_crop_df(subset_df, split)
    raise ValueError(f'Unsupported BBOX_SOURCE: {BBOX_SOURCE}')

train_crop_df = resolve_crop_source_df(train_df, 'train')
val_crop_df = resolve_crop_source_df(val_df, 'validation')

CROP_MARGIN_RATIO = 0.20
PADDING_MODE = 'reflect'
SAVE_EXT = '.png'

PAD_MODE_MAP = {
    'reflect': cv2.BORDER_REFLECT_101,
    'edge': cv2.BORDER_REPLICATE,
    'constant': cv2.BORDER_CONSTANT,
}

assert PADDING_MODE in PAD_MODE_MAP

def read_rgb(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(image_path)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def bbox_to_xyxy(row):
    x1 = float(row['bbox_x'])
    y1 = float(row['bbox_y'])
    x2 = x1 + float(row['bbox_w'])
    y2 = y1 + float(row['bbox_h'])
    return x1, y1, x2, y2

def build_square_crop_box(row, margin_ratio=CROP_MARGIN_RATIO):
    x1, y1, x2, y2 = bbox_to_xyxy(row)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    side = max(x2 - x1, y2 - y1) * (1.0 + margin_ratio)
    return cx - side / 2.0, cy - side / 2.0, cx + side / 2.0, cy + side / 2.0

def crop_square_from_original(image, row, padding_mode=PADDING_MODE):
    img_h, img_w = image.shape[:2]
    crop_x1, crop_y1, crop_x2, crop_y2 = build_square_crop_box(row)

    pad_left = max(0, int(math.ceil(-crop_x1)))
    pad_top = max(0, int(math.ceil(-crop_y1)))
    pad_right = max(0, int(math.ceil(crop_x2 - img_w)))
    pad_bottom = max(0, int(math.ceil(crop_y2 - img_h)))

    padded = cv2.copyMakeBorder(
        image,
        pad_top,
        pad_bottom,
        pad_left,
        pad_right,
        borderType=PAD_MODE_MAP[padding_mode],
        value=[0, 0, 0],
    )

    crop_x1 += pad_left
    crop_x2 += pad_left
    crop_y1 += pad_top
    crop_y2 += pad_top

    x1i = int(np.floor(crop_x1))
    y1i = int(np.floor(crop_y1))
    x2i = int(np.ceil(crop_x2))
    y2i = int(np.ceil(crop_y2))

    crop = padded[y1i:y2i, x1i:x2i]
    if crop.size == 0:
        raise ValueError('Empty crop generated')

    raw_h, raw_w = crop.shape[:2]

    crop_side = float(max(crop_x2 - crop_x1, crop_y2 - crop_y1))
    pad_fraction = (pad_left + pad_right + pad_top + pad_bottom) / max(crop_side * 4.0, 1.0)

    meta = {
        'bbox_x1': round(float(row['bbox_x']), 4),
        'bbox_y1': round(float(row['bbox_y']), 4),
        'bbox_x2': round(float(row['bbox_x']) + float(row['bbox_w']), 4),
        'bbox_y2': round(float(row['bbox_y']) + float(row['bbox_h']), 4),
        'crop_side_px': round(crop_side, 4),
        'raw_h': int(raw_h),
        'raw_w': int(raw_w),
        'pad_left': pad_left,
        'pad_right': pad_right,
        'pad_top': pad_top,
        'pad_bottom': pad_bottom,
        'pad_fraction': round(pad_fraction, 6),
    }
    return crop, meta


## 5. Crop 정책

- bbox 타이트 crop 대신 정사각형 margin crop
- 경계 밖으로 나가면 reflect padding
- crop 단계에서는 raw square crop만 저장
- 최종 classifier 입력 resize는 dataloader/transform 단계에서 수행
- `BBOX_SOURCE='yolo'`이면 detector 오차가 반영된 실제 파이프라인 crop

## 6. Crop 생성 및 manifest 저장

- 저장 경로: `/content/crops/{split}/{item_seq}/...png`
- resize 전 raw crop을 그대로 저장
- 이후 classifier 학습 시 `item_seq`를 class label로 사용 가능
- crop metadata csv도 함께 저장해서 padding/원본 crop 크기 분석 가능


In [ ]:
def build_crop_dataset(crop_df, split):
    src_dir = Path(f"/content/raw_images/{split}")
    out_root = Path(f"/content/crops/{split}")
    out_root.mkdir(parents=True, exist_ok=True)

    records = []
    missing_images = 0

    for ann_idx, row in tqdm(crop_df.reset_index(drop=True).iterrows(), total=len(crop_df), desc=f"{split} crops"):
        image_path = src_dir / row['image_file']
        if not image_path.exists():
            missing_images += 1
            continue

        image = read_rgb(image_path)
        crop, meta = crop_square_from_original(image, row)

        class_dir = out_root / str(row['item_seq'])
        class_dir.mkdir(parents=True, exist_ok=True)

        crop_name = f"{Path(row['image_file']).stem}_{row['source_bbox']}_ann{ann_idx:06d}{SAVE_EXT}"
        crop_path = class_dir / crop_name
        Image.fromarray(crop).save(crop_path)

        records.append({
            'split': split,
            'item_seq': row['item_seq'],
            'image_file': row['image_file'],
            'crop_file': crop_name,
            'crop_path': str(crop_path.relative_to('/content')),
            'label_zip_name': row['label_zip_name'],
            'source_bbox': row['source_bbox'],
            'pred_conf': row.get('pred_conf', np.nan),
            **meta,
        })

    crop_manifest = pd.DataFrame(records)
    manifest_path = Path(f"/content/crops_{split}_manifest.csv")
    crop_manifest.to_csv(manifest_path, index=False)

    print(f"{split}: saved {len(crop_manifest):,} crops")
    print(f"{split}: missing source images {missing_images:,}")
    display(crop_manifest.head(3))
    return crop_manifest

train_crop_manifest = build_crop_dataset(train_crop_df, 'train')
val_crop_manifest = build_crop_dataset(val_crop_df, 'validation')

print("\n--- train class counts (top 10) ---")
display(train_crop_manifest['item_seq'].value_counts().head(10))

print("\n--- padding summary ---")
display(pd.DataFrame({
    'train': train_crop_manifest['pad_fraction'].describe(),
    'validation': val_crop_manifest['pad_fraction'].describe(),
}))


## 7. 샘플 QC

- crop이 지나치게 타이트한지
- reflect padding이 얼마나 자주 발생하는지
- item_seq별로 시각 품질이 유지되는지 확인


In [ ]:
def show_samples(crop_manifest, n=12, seed=42):
    sample = crop_manifest.sample(n=min(n, len(crop_manifest)), random_state=seed).reset_index(drop=True)
    cols = 4
    rows = int(np.ceil(len(sample) / cols))

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, rec in sample.iterrows():
        ax = plt.subplot(rows, cols, i + 1)
        img = Image.open(Path('/content') / rec['crop_path'])
        ax.imshow(img)
        ax.set_title(f"item_seq={rec['item_seq']}\n{rec['source_bbox']}\npad={rec['pad_fraction']:.3f}")
        ax.axis('off')
    plt.tight_layout()

show_samples(train_crop_manifest, n=12)


## 8. 선택: GCS 저장

- 기본값은 업로드 비활성화
- crop 결과와 manifest를 필요할 때만 저장


In [ ]:
EXPORT_TO_GCS = False
GCS_OUT = 'gs://pilliot-raw-data-2026/05_crop_baselines/jh_crop_baseline'

if EXPORT_TO_GCS:
    os.system(f'gcloud storage cp -r /content/crops "{GCS_OUT}/crops"')
    os.system(f'gcloud storage cp /content/crops_train_manifest.csv /content/crops_validation_manifest.csv "{GCS_OUT}/"')
    print('GCS 저장 완료')
else:
    print('EXPORT_TO_GCS=False -> upload skipped')
